In [6]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report

# Download NLTK stopwords quietly
nltk.download('stopwords', quiet=True)
stop_words = set(stopwords.words('english'))

# ---------------------------------------------------------
# 1. Load Dataset Directly from URL
# ---------------------------------------------------------
print("1. Downloading dataset...")
url = "https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv"
df = pd.read_csv(url)

# ---------------------------------------------------------
# 2. Text Preprocessing using NLTK and Regex
# ---------------------------------------------------------
print("2. Cleaning text data (this takes a few seconds)...")
def clean_text(text):
    if not isinstance(text, str):
        return ""
    # Remove punctuation and convert to lowercase
    text = re.sub(r'[^\w\s]', '', text).lower()
    # Remove stopwords
    return ' '.join([word for word in text.split() if word not in stop_words])

df['clean_text'] = df['text'].apply(clean_text)

# ---------------------------------------------------------
# 3. Train-Test Split
# ---------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label'], test_size=0.2, random_state=42
)

# ---------------------------------------------------------
# 4. Vectorize Text (TF-IDF)
# ---------------------------------------------------------
print("3. Vectorizing text...")
tfidf = TfidfVectorizer(max_df=0.7) # Ignore words appearing in >70% of articles
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# ---------------------------------------------------------
# 5. Fast Model Training
# ---------------------------------------------------------
print("4. Training Passive Aggressive Classifier...")
pac = PassiveAggressiveClassifier(max_iter=50)
pac.fit(X_train_tfidf, y_train)

# ---------------------------------------------------------
# 6. Evaluation
# ---------------------------------------------------------
y_pred = pac.predict(X_test_tfidf)
print("\n========== MODEL EVALUATION ==========")
print(f"Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("======================================\n")

# ---------------------------------------------------------
# 7. Mini App Interface
# ---------------------------------------------------------
def detect_fake_news(news_text):
    """Pass a string of text to see if the model thinks it is FAKE or REAL."""
    cleaned = clean_text(news_text)
    vectorized = tfidf.transform([cleaned])
    prediction = pac.predict(vectorized)[0]
    return prediction

# Test the app with custom inputs
test_article_1 = input("enter the news : ")

print("--- Testing the Mini App ---")
print(f"Input 1: '{test_article_1}'")
print(f"Prediction: {detect_fake_news(test_article_1)}\n")

print(f"Input 2: '{test_article_2}'")
print(f"Prediction: {detect_fake_news(test_article_2)}")

1. Downloading dataset...
2. Cleaning text data (this takes a few seconds)...
3. Vectorizing text...
4. Training Passive Aggressive Classifier...

========== MODEL EVALUATION ==========
Accuracy: 93.92%

Classification Report:
              precision    recall  f1-score   support

        FAKE       0.94      0.93      0.94       628
        REAL       0.94      0.94      0.94       639

    accuracy                           0.94      1267
   macro avg       0.94      0.94      0.94      1267
weighted avg       0.94      0.94      0.94      1267


enter the news : SpaceX successfully launched its latest batch of Starlink satellites into low Earth orbit early this morning. The Falcon 9 first-stage booster landed safely on the drone ship stationed in the Atlantic Ocean, marking the company's 50th successful recovery this year. y
--- Testing the Mini App ---
Input 1: 'SpaceX successfully launched its latest batch of Starlink satellites into low Earth orbit early this morning. The Falcon 

In [9]:
# ---------------------------------------------------------
# 1. Generate requirements.txt for Streamlit Cloud
# ---------------------------------------------------------
with open("requirements.txt", "w") as f:
    f.write('''streamlit>=1.30.0
pandas>=2.0.0
scikit-learn>=1.0.0
nltk>=3.8.0
''')

# ---------------------------------------------------------
# 2. Generate the production-ready app.py
# ---------------------------------------------------------
with open("app.py", "w") as f:
    f.write('''import streamlit as st
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier

# Page layout styling
st.set_page_config(page_title="Fake News Detector", page_icon="📰", layout="centered")

# Download NLTK data quietly inside the Streamlit cache framework
@st.cache_resource
def load_nlp_resources():
    nltk.download('stopwords', quiet=True)
    return set(stopwords.words('english'))

stop_words = load_nlp_resources()

# Load dataset and train the model (Cached so it only runs once upon app startup)
@st.cache_resource
def train_model():
    # Dataset URL
    url = "https://raw.githubusercontent.com/lutzhamel/fake-news/master/data/fake_or_real_news.csv"
    df = pd.read_csv(url)

    # Text cleaning helper logic
    def clean_text(text):
        if not isinstance(text, str):
            return ""
        text = re.sub(r'[^\w\s]', '', text).lower()
        return ' '.join([word for word in text.split() if word not in stop_words])

    df['clean_text'] = df['text'].apply(clean_text)

    # Dataset splitting
    X_train, X_test, y_train, y_test = train_test_split(
        df['clean_text'], df['label'], test_size=0.2, random_state=42
    )

    # Vectorization & Fast Passive Aggressive Training
    tfidf = TfidfVectorizer(max_df=0.7)
    X_train_tfidf = tfidf.fit_transform(X_train)

    pac = PassiveAggressiveClassifier(max_iter=50)
    pac.fit(X_train_tfidf, y_train)

    return tfidf, pac, clean_text

# Instantiate the built model assets
tfidf, pac, clean_text = train_model()

# --- STREAMLIT UI ---
st.title("📰 Fake News Detector")
st.markdown("This mini-app utilizes **TF-IDF Feature Extraction** paired with a **Passive Aggressive Classifier** trained natively via `scikit-learn` to analyze the factual integrity of political text strings.")

st.subheader("Analyze News Content")
user_input = st.text_area("Paste the text block of a news article below:", height=250, placeholder="Insert content here...")

if st.button("Evaluate Text", type="primary"):
    if not user_input.strip():
        st.warning("Please input valid text content to begin analysis.")
    else:
        with st.spinner("Analyzing linguistic features..."):
            # Process & predict
            processed_input = clean_text(user_input)
            vector_input = tfidf.transform([processed_input])
            prediction = pac.predict(vector_input)[0]

            st.write("---")
            if prediction == "REAL":
                st.success("### ✅ Prediction: This content maps closely to factual **REAL** news indicators.")
            else:
                st.error("### 🚨 Prediction: This content maps closely to fabricated **FAKE** news indicators.")
''')

print("Successfully generated 'app.py' and 'requirements.txt'!")
print("Download both files from the Colab file panel (left sidebar folder icon) to deploy them to GitHub.")

Successfully generated 'app.py' and 'requirements.txt'!
Download both files from the Colab file panel (left sidebar folder icon) to deploy them to GitHub.


<>:46: SyntaxWarning: invalid escape sequence '\w'
<>:46: SyntaxWarning: invalid escape sequence '\w'
/tmp/ipykernel_1243/2024515521.py:46: SyntaxWarning: invalid escape sequence '\w'
  text = re.sub(r'[^\w\s]', '', text).lower()
